# Forecast-vintage macro: feature completeness over history

Uses `ForecastVintageMacroStore` (same loader as `expanding_window(realtime=True)`).

## Vintage handling (important)

Each call `panel_for_forecast_date(t, …)` uses **`select_vintage_for_forecast_date(t)`**: the ALFRED as-of date and archive fallback are chosen **for that forecast month `t`**, not using “the last vintage everywhere”.

**Measures 2 and 3** therefore scan **one panel per month** over the sample (647 calls for 1971-08–2025-06): every month gets its own information set, same logic as the slow loop in `expanding_window(realtime=True)`.

**Measure 1** is different on purpose: it builds **only one** panel at `END`, i.e. one fixed information set applied to the whole `START…END` range. That is a **retrospective sanity check**, not pseudo real time. Ignore it if you only care about vintage-correct coverage.

---

**Measures** (transformed macro after `prepare_macro_panel_for_project`; 124 series after dropping `TWEXAFEGSMTHx` and `ACOGNO`):

1. **(Optional) Latest-as-of at END** — Single panel `panel_for_forecast_date(END, start=START, end=END)`. Not vintage-correct through time; see above.
2. **Stitched contemporaneous rows** — For **each** `t`, take month `t`’s row from `panel_for_forecast_date(t, end=t)` (correct vintage for `t`). Column complete if that row is never missing across all `t`.
3. **Strict expanding-window** — For **each** `t`, require the column to be non-missing on **all** rows `START…t` inside **`panel_for_forecast_date(t, end=t)`**. Intersect over every `t`. This matches `drop_unavailable_columns` on training **before** `ffill`, when training always begins at `START`.

Default `START` / `END` match `orchestrator_linear_runs.ipynb`. The per-month loop is slow (~minutes).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.forecast_vintages import ForecastVintageMacroStore, month_end

START = "1971-08-31"
END = "2025-06-30"

store = ForecastVintageMacroStore()
month_idx = pd.date_range(START, END, freq="M")
print(f"Months: {len(month_idx)}")

In [ ]:
def report_complete(name: str, df: pd.DataFrame) -> pd.Series:
    ok = df.notna().all(axis=0)
    n = int(ok.sum())
    print(f"\n=== {name} ===")
    print(f"shape: {df.shape} | complete columns: {n} / {df.shape[1]}")
    bad = ok[~ok].index.tolist()
    if bad:
        print(f"Incomplete ({len(bad)}): {bad}")
    return ok


panel_end = store.panel_for_forecast_date(END, start=START, end=END)
macro_end = panel_end.transformed
print(f"Transformed macro columns: {macro_end.shape[1]}")
ok_latest = report_complete(
    "1) (Optional) Single panel at END — not vintage-correct through time",
    macro_end,
)

In [ ]:
rows = []
strict_intersection = None

# Each iteration: new vintage via select_vintage_for_forecast_date(d) inside the store.
for d in tqdm(month_idx, desc="per-month panels"):
    panel = store.panel_for_forecast_date(d, start=START, end=d)
    X = panel.transformed
    dme = month_end(d)
    rows.append(X.loc[[dme]])
    ok_train = X.notna().all(axis=0)
    strict_intersection = ok_train if strict_intersection is None else (strict_intersection & ok_train)

stitched = pd.concat(rows).sort_index()
ok_stitch = report_complete("2) Stitched contemporaneous rows", stitched)

n_strict = int(strict_intersection.sum())
print(f"\n=== 3) Strict expanding-window (intersection over all train ends) ===")
print(f"complete columns: {n_strict} / {len(strict_intersection)}")
bad = strict_intersection[~strict_intersection].index.tolist()
if bad:
    print(f"Incomplete ({len(bad)}): {bad}")

In [ ]:
summary = pd.DataFrame(
    [
        {"measure": "single_panel_at_end_not_expanding", "complete_columns": int(ok_latest.sum()), "total_columns": macro_end.shape[1]},
        {"measure": "stitched_contemporaneous", "complete_columns": int(ok_stitch.sum()), "total_columns": stitched.shape[1]},
        {"measure": "strict_expanding_intersection", "complete_columns": n_strict, "total_columns": len(strict_intersection)},
    ]
)
summary